# 02 — 一本脚ホッパーを MuJoCo で動かす

2D シミュレーター（`tools/webapp/hopper-2d.html` / `hopper-roll-2d.html`）で成立した機体と Raibert 制御を、
そのまま 3D の MuJoCo に移す。**GPU は不要**（CPU ランタイムでよい）。学習はしない。

- モデル生成: `hopper/model.py`（胴体フリージョイント＋ロール軸＋5 節閉ループ＋直列バネ＋点接地。サーボはトルク─速度直線＋反射慣性）
- 制御: `hopper/controller.py`（2D と同じ Raibert 3 分割。接地中は電流指令、飛行中は位置 PD）
- 実行: `hopper/run.py`（指標と動画）

目的は 2D の数字（頂点 5 cm、接地 165 ms、脚長サーボ 56 / 77 %、前進 0.2 m/s、横 0.2 m/s）が 3D でも出るかと、
2D では扱えなかった**ヨーのドリフト**と**ピッチ×ロールの連成**を見ること。結果は HANDOFF §8.10 に記録する。


In [ ]:
#@title 1. リポジトリ取得
REPO_URL = "https://github.com/yosihitoyasudasub/quadleg-rl.git"  #@param {type:"string"}
REPO_DIR = "/content/quadleg-rl"
import os, sys, importlib
if os.path.isdir(REPO_DIR):
    !cd $REPO_DIR && git pull
else:
    !git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()


In [ ]:
#@title 2. インストール（mujoco と mediapy だけ。再起動は不要なことが多い）
%pip install -q -U mujoco mediapy
import os
os.environ["MUJOCO_GL"] = "egl"      # Colab のオフスクリーン描画
import mujoco, mediapy, numpy as np
print("mujoco", mujoco.__version__)


In [ ]:
#@title 3. モデルを組んで静的確認
import importlib, hopper.model, hopper.kinematics, hopper.controller, hopper.run
for m in (hopper.kinematics, hopper.model, hopper.controller, hopper.run):
    importlib.reload(m)
from hopper.model import HopperParams, build_xml, initial_qpos
from hopper.run import make
p, model, data = make(HopperParams())
print("nq", model.nq, "nv", model.nv, "nu", model.nu, "timestep", model.opt.timestep)
print("qpos0", np.round(data.qpos, 3))
print("body mass total", model.body_subtreemass[model.body('torso').id], "kg")
# 1 コマ描画（初期姿勢）
os.environ["MUJOCO_GL"] = "egl"
r = mujoco.Renderer(model, height=360, width=640)
cam = mujoco.MjvCamera(); cam.type = mujoco.mjtCamera.mjCAMERA_FREE
cam.distance, cam.azimuth, cam.elevation = 1.0, 135, -15; cam.lookat[:] = [0, 0, 0.15]
r.update_scene(data, camera=cam); mediapy.show_image(r.render()); r.close()


In [ ]:
#@title 4. その場ホップ（15 s）— 2D の期待値: 頂点 5.0 cm、Ts 165 ms、脚長サーボ τ 56 % / ω 77 %
from hopper.run import simulate, hop_table
res = simulate(duration=15.0)
print(hop_table(res))


In [ ]:
#@title 5. 外乱からの回復（前進 0.3 m/s ＋ ピッチ 0.1 rad、横 0.3 m/s ＋ ロール 0.1 rad）
res_p = simulate(duration=15.0, vx0=0.3, th0=0.1)
res_r = simulate(duration=15.0, vy0=0.3, roll0=0.1)


In [ ]:
#@title 6. 前進 0.2 m/s と 横 0.2 m/s（2D の期待値: 0.16〜0.19 / 0.23 m/s）
res_fwd = simulate(duration=20.0, vx_des=0.2)
print(hop_table(res_fwd, 8))
res_lat = simulate(duration=20.0, vy_des=0.2)
print(hop_table(res_lat, 8))
# ヨーのドリフト（2D では見えなかった項目）
import math
print("yaw drift per hop [deg]:", [round(math.degrees(h['yaw']),1) for h in res_fwd['hops'][-8:]])


In [ ]:
#@title 7. 動画（前進 0.2 m/s、8 s）
res_v = simulate(duration=8.0, vx_des=0.2, record=True, fps=50)
mediapy.show_video(res_v["frames"], fps=50)


In [ ]:
#@title 8. パラメータを変えて試す（例: 目標 8 cm、l1=100）
from dataclasses import replace
from hopper.model import HopperParams
from hopper.controller import ControlGains
p2 = HopperParams(l1=0.100, l2=0.120)
g2 = ControlGains(hdes=0.08)
res2 = simulate(duration=15.0, params=p2, gains=g2)


## トラブルシューティング

- `Renderer` でエラー → セル 2 の `MUJOCO_GL=egl` が `import mujoco` より前に設定されているか確認。だめなら `osmesa`。
- 初期姿勢で脚が組めない（`初期姿勢に到達できません`）→ `L0 − retract − ls0` が `l1 + l2` と `|l1 − l2|` の間にあるか。
- 2D と数字が合わない → まず `hop_table` で接地時間 Ts と蹴り出し dL を比べる。Ts が大きく違えばバネ・質量の写し間違い、dL が上限 50 mm に張り付いていれば着地エネルギーの回収が減っている（底付き・減衰）。
- 転倒する → `ControlGains` の `cn`/`kv` は 2D の接地写像から決めた値（0.5 / 0.03）。3D で写像が変わっていれば、2D と同じ強制実験（`ctl.xf_cmd` を上書き）で測り直す。
